In [ ]:
from citibikes.cityibikes_utils import get_trip_duration
from utility.datetime_utility import timestamp_to_date
import pyspark.sql.functions as F

In [ ]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.table(f'{catalog}.01_bronze.jc_citibikes')

In [5]:
df = get_trip_duration(spark, df, 'started_at', 'ended_at', 'trip_duration_mins')

In [6]:
df = timestamp_to_date(spark, df, 'started_at', 'trip_date')

In [ ]:
df = df.withColumn('metadata',
                   F.create_map(
                     F.lit("pipeline_id"), F.lit(pipeline_id),
                     F.lit("run_id"), F.lit(run_id),
                     F.lit("task_id"), F.lit(task_id),
                     F.lit("processed_timestamp"), F.lit(processed_timestamp))
)

In [ ]:
df.writeTo(f"{catalog}.02_silver.jc_citibikes").createOrReplace()